## Problems
**Q1:** Design a data structure that:
- `boolean insert(int val)` inserts an item value into the set if not present. Returns true if the item was not present, false otherwise.
- `boolean remove(int val)` removes an item val from the set if present. Returns true if the item was present, false otherwise.
- `int getRandom()` returns a random element from the current set of elements (it's guaranteed that at least one element exists when this method is called). Each element must have the same probability of being returned.

All the above operations should complete in $O(1)$ time.  

**Answer:** store the values in a arraylist so you can access a random element in $O(1)$ time. Also store it in a map so that you can add and remove in $O(1)$.

In [ ]:
class RandomizedSet {
    private final List<Integer> list;
    private final Map<Integer, Integer> map;  // Key is the value being stored and value is its index in the above list

    public RandomizedSet() {
        map = new HashMap<>();
        list = new ArrayList<>();
    }

    public boolean insert(int val) {
        if (map.containsKey(val)) {
            return false;
        }

        list.add(val);
        map.put(val, list.size() - 1);
        return true;
    }

    public boolean remove(int val) {
        Integer index = map.remove(val);
        if (index == null) {
            return false;
        }

        int lastNum = list.getLast();
        int lastIndex = list.size() - 1;

        // Takes care of the case where you are removing the last value
        if (index != lastIndex) {
            list.set(index, lastNum);
            map.put(lastNum, index);
        }

        list.removeLast();
        return true;
    }

    public int getRandom() {
        int randomIndex = ThreadLocalRandom.current().nextInt(0, list.size());
        return list.get(randomIndex);
    }
}

What happens if we want to allow duplicates to be inserted?

**Q 2:** Design a stack class that allows the following operations:
- CustomStack(int maxSize) Initializes the object with maxSize which is the maximum number of elements in the stack.
- `void push(int x)` adds x to the top of the stack if the stack has not reached the maximum size.
- `int pop()` pops and returns the top of the stack or -1 if the stack is empty.
- `void inc(int k, int val)` increments the bottom k elements of the stack by val. If there are less than k elements in the stack, increment all the elements in the stack.

We need to perform all these operations in $O(1)$ time complexity.

**Answer:**

In [ ]:
class Stack {
    private ArrayDeque<Integer> stack;
    // All the elements in stack <= ith index of operations
    // should be added with operations[i]
    private List<Integer> operations;
    private int maxSize;
    private int occupancy;

    public Stack(int maxSize) {
        this.maxSize = maxSize;
        stack = new ArrayDeque<>();
        operations = new ArrayList<>();
    }

    public void push(int x) {
        if (occupancy < maxSize) {
            stack.push(x);
            operations.add(0);

            occupancy++;
        }
    }

    public int pop() {
        if (occupancy == 0) {
            return -1;
        }
        
        int value =  stack.pop();
        int temp = operations.removeLast();
        value += temp;
        
        if (!operations.isEmpty())
            operations.set(operations.size() - 1, operations.getLast() + temp);
        
        occupancy--;
        
        return value;
    }

    public void increment(int k, int val) {
        int index = Math.min(k, stack.size()) - 1;

        if (index >= 0) operations.set(index, operations.get(index) + val);
    }
}

**Q 3:** Design a data structure to store the count of keys. It supports:
- `inc(String key)` increments the count of the string key by 1. If key does not exist in the data structure, insert it with count 1.
- `dec(String key)` decrements the count of the string key by 1. If the count of key is 0 after the decrement, remove it from the data structure. It is guaranteed that key exists in the data structure before the decrement.
- `getMaxKey()` returns one of the keys with the maximal count. If no element exists, return an empty string "".
- `getMinKey()` returns one of the keys with the minimum count. If no element exists, return an empty string "".

We need to complete all these operations in $O(1)$ time.  
**Answer:**

In [ ]:
class AllOne {
    static class Node {
        int freq;
        Set<String> keys;

        Node prev;
        Node next;

        Node(int freq) {
            this.freq = freq;
            this.keys = new HashSet<>();
        }
    }

    private final Map<String, Node> map;

    private final Node head;
    private final Node tail;

    public AllOne() {
        map = new HashMap<>();

        head = new Node(0);
        tail = new Node(0);

        head.next = tail;
        tail.prev = head;
    }

    public void inc(String key) {
        // Check if it is a new key.
        if (!map.containsKey(key)) {

            Node node;
            // Get the frequency node for frequency 1
            // Frequency 1 bucket already exists
            if (head.next != tail && head.next.freq == 1) {
                node = head.next;
            // Frequency 1 bucket doesn't exist, so we create one
            } else {
                node = new Node(1);
                insertAfter(head, node);
            }

            node.keys.add(key);
            map.put(key, node);
            return;
        }

        // Key already exists, so we get the frequency node
        // associated with the key
        Node curr = map.get(key);
        int newFreq = curr.freq + 1;

        Node nextNode;
        // Frequency node for the new frequency already exists, get that
        if (curr.next != tail && curr.next.freq == newFreq) {
            nextNode = curr.next;
        // Frequency node doesn't exist, so we create one
        } else {
            nextNode = new Node(newFreq);
            insertAfter(curr, nextNode);
        }

        nextNode.keys.add(key);
        curr.keys.remove(key);
        map.put(key, nextNode);

        // If the old node has no keys left, remove it
        if (curr.keys.isEmpty()) {
            remove(curr);
        }
    }

    public void dec(String key) {
        // Check if the key exists in the map or not
        if (!map.containsKey(key)) {
            return;
        }

        // Get the frequency node of the current key
        Node curr = map.get(key);

        // If the current frequency is 1, then the key needs to be completely removed
        if (curr.freq == 1) {
            curr.keys.remove(key);
            map.remove(key);

            // Is the frequency node now empty?
            if (curr.keys.isEmpty()) {
                remove(curr);
            }

            return;
        }

        int newFreq = curr.freq - 1;

        Node prevNode; 
        // Node for the new frequency already exists, use it
        if (curr.prev != head && curr.prev.freq == newFreq) {
            prevNode = curr.prev;
        // Node for the new frequency doesn't exist, we need to create it
        } else {
            prevNode = new Node(newFreq);
            insertAfter(curr.prev, prevNode);
        }

        prevNode.keys.add(key);
        curr.keys.remove(key);
        map.put(key, prevNode);

        if (curr.keys.isEmpty()) {
            remove(curr);
        }
    }

    public String getMaxKey() {
        if (tail.prev == head) {
            return "";
        }

        return tail.prev.keys.iterator().next();
    }

    public String getMinKey() {
        if (head.next == tail) {
            return "";
        }

        return head.next.keys.iterator().next();
    }

    private void insertAfter(Node prev, Node node) {
        node.next = prev.next;
        node.prev = prev;

        prev.next.prev = node;
        prev.next = node;
    }

    private void remove(Node node) {
        node.prev.next = node.next;
        node.next.prev = node.prev;
    }
}

**Q 4:** Design an LRU cache.  
**Answer:**

In [ ]:
class LRUCache {
    private final int CAPACITY;
    private int occupancy;
    private final Map<Integer, Node> cache;
    private final Node head;
    private final Node tail;

    public LRUCache(int capacity) {
        this.CAPACITY = capacity;
        this.cache = new HashMap<>();
        this.head = new Node(-1, -1);
        this.tail = new Node(-1, -1);
        this.head.next = this.tail;
        this.tail.prev = this.head;
    }

    public int get(int key) {
        // Check if the key is in the cache
        if (cache.containsKey(key)) {
            Node node = cache.get(key);
            remove(node);
            append(node);
            return node.value;
        }

        return -1;
    }

    public void put(int key, int value) {
        // If there is no space in the cache, remove the least recent entry
        if (occupancy == CAPACITY && !cache.containsKey(key)) {
            Node least = head.next;
            remove(least);
            cache.remove(least.key);
            occupancy--;
        }

        // Check if the key is in the cache
        if (cache.containsKey(key)) {
            Node node = cache.get(key);
            remove(node);
            append(node);
            node.value = value;
        } else {
            Node node = new Node(key, value);
            append(node);
            cache.put(key, node);
            occupancy++;
        }
    }

    private void remove(Node node) {
        Node prev = node.prev;
        Node next = node.next;
        prev.next = next;
        next.prev = prev;
    }

    private void append(Node node) {
        Node prev = tail.prev;
        prev.next = node;
        node.prev = prev;
        node.next = tail;
        tail.prev = node;
    }

    static class Node {
        public int key;
        public int value;
        public Node prev;
        public Node next;

        public Node(int key, int value) {
            this.key = key;
            this.value = value;
        }
    }
}

**Q5:** Design an LFU cache.  
**Answer:**